In [2]:
import pandas as pd
import numpy as np

# 1. Load the training/testing dataset
train_df = pd.read_csv("train-test.csv")

# 2. Check dataset shape and data types
print("--- Dataset Shape ---")
print(train_df.shape)

print("\n--- Columns and Data Types ---")
print(train_df.dtypes)

print("\n--- Missing Values Count ---")
print(train_df.isnull().sum())

# 3. Check for negative weights (data quality issue)
negative_weights_count = (train_df['weight'] < 0).sum()
print(f"\nNumber of rows with negative weights: {negative_weights_count}")

--- Dataset Shape ---
(16878, 14)

--- Columns and Data Types ---
load_id          object
pickup           object
delivery         object
pickup_lat      float64
pickup_lon      float64
delivery_lat    float64
delivery_lon    float64
distance        float64
equipment        object
weight          float64
date             object
market_index    float64
quote_signal    float64
posted_rate     float64
dtype: object

--- Missing Values Count ---
load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          116
date              0
market_index    140
quote_signal      1
posted_rate       1
dtype: int64

Number of rows with negative weights: 116


In [3]:

# Clean weights: convert negative weights to positive using abs(), then fill missing with median
train_df['weight'] = train_df['weight'].abs()
train_df['weight'] = train_df['weight'].fillna(train_df['weight'].median())

# Fill missing market_index with median
train_df['market_index'] = train_df['market_index'].fillna(train_df['market_index'].median())

# Drop rows with missing target (posted_rate) or quote_signal if any
train_df = train_df.dropna(subset=['posted_rate', 'quote_signal'])

# Convert date to datetime and extract temporal features
train_df['date'] = pd.to_datetime(train_df['date'])
train_df['month'] = train_df['date'].dt.month
train_df['dayofweek'] = train_df['date'].dt.dayofweek
train_df['dayofyear'] = train_df['date'].dt.dayofyear

print("Data cleaning completed successfully!")
print("New dataset shape:", train_df.shape)

Data cleaning completed successfully!
New dataset shape: (48000, 17)


In [4]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Select features and target variable
features = [
    'distance', 'weight', 'market_index', 'quote_signal',
    'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon',
    'month', 'dayofweek', 'dayofyear'
]
target = 'posted_rate'

X = train_df[features]
y = train_df[target]

# 2. Split data for local validation (train/test split)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train the model
model = HistGradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)

# 4. Evaluate the model
preds = model.predict(X_val)
mae = mean_absolute_error(y_val, preds)
rmse = np.sqrt(mean_squared_error(y_val, preds))

print(f"Validation MAE: {mae:.2f}")
print(f"Validation RMSE: {rmse:.2f}")

Validation MAE: 140.01
Validation RMSE: 543.53


In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Load datasets
train_df = pd.read_csv("train-test.csv")
val_df = pd.read_csv("validation.csv")
dec_df = pd.read_csv("december-chart-inputs.csv")

# 2. Clean training data
train_df['weight'] = train_df['weight'].abs()
train_df['weight'] = train_df['weight'].fillna(train_df['weight'].median())
train_df['market_index'] = train_df['market_index'].fillna(train_df['market_index'].median())
train_df['date'] = pd.to_datetime(train_df['date'])
train_df['month'] = train_df['date'].dt.month
train_df['dayofweek'] = train_df['date'].dt.dayofweek
train_df['dayofyear'] = train_df['date'].dt.dayofyear

# 3. Train the model on full train data
features = [
    'distance', 'weight', 'market_index', 'quote_signal',
    'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon',
    'month', 'dayofweek', 'dayofyear'
]
target = 'posted_rate'

model = HistGradientBoostingRegressor(random_state=42)
model.fit(train_df[features], train_df[target])

# 4. Clean and predict validation data (12,000 loads)
val_clean = val_df.copy()
val_clean['weight'] = val_clean['weight'].abs()
val_clean['weight'] = val_clean['weight'].fillna(train_df['weight'].median())
val_clean['market_index'] = val_clean['market_index'].fillna(train_df['market_index'].median())
val_clean['date'] = pd.to_datetime(val_clean['date'])
val_clean['month'] = val_clean['date'].dt.month
val_clean['dayofweek'] = val_clean['date'].dt.dayofweek
val_clean['dayofyear'] = val_clean['date'].dt.dayofyear

val_preds = model.predict(val_clean[features])

# Save to validation_predictions.csv using the template
sub = pd.read_csv('validation-predictions-template.csv')
sub['predicted_rate'] = val_preds
sub.to_csv('validation_predictions.csv', index=False)

# 5. Predict December chart inputs
dec_clean = dec_df.copy()
dec_clean['date'] = pd.to_datetime(dec_clean['date'])
dec_clean['month'] = dec_clean['date'].dt.month
dec_clean['dayofweek'] = dec_clean['date'].dt.dayofweek
dec_clean['dayofyear'] = dec_clean['date'].dt.dayofyear

# Fill missing lats/lons and market/quote signals using median values from train
dec_clean['pickup_lat'] = 36.991520
dec_clean['pickup_lon'] = -84.998760
dec_clean['delivery_lat'] = 41.315610
dec_clean['delivery_lon'] = -85.362060
dec_clean['market_index'] = train_df['market_index'].median()
dec_clean['quote_signal'] = train_df['quote_signal'].median()

dec_preds = model.predict(dec_clean[features])
dec_clean['predicted_rate'] = dec_preds

# Save December inputs back
dec_clean[['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']].to_csv('december-chart-inputs.csv', index=False)

print("All predictions generated successfully!")

All predictions generated successfully!


In [7]:
import subprocess

result = subprocess.run([
    'python', 'score.py',
    '--predictions', 'validation_predictions.csv',
    '--december-predictions', 'december-chart-inputs.csv'
], capture_output=True, text=True)

print("Return code:", result.returncode)
print("Stdout:", result.stdout)
print("Stderr:", result.stderr)

Return code: 0
Stdout: Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results/candidate_december.png
Final validation metrics are calculated by Spotter after submission.

Stderr: 
